<a href="https://colab.research.google.com/github/zeynepdnnz/cs445-semeval-task5/blob/main/CS445_STS_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup and Imports**

In [1]:
!pip install -q "transformers>=4.40,<4.45"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/')

In [3]:
import pandas
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, BatchEncoding,
    DataCollatorWithPadding, EvalPrediction,
    EarlyStoppingCallback
    )
from datasets import Dataset, DatasetDict, load_dataset, Value
import numpy as np
from scipy.stats import spearmanr

In [4]:
model_id = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [5]:
def input_format(sample: pandas.Series) -> tuple[str, str]:
  story_part = [sample['precontext'], sample['sentence'], sample['ending'] or '']
  story_part = " ".join(story_part)
  meaning_part = (f"{sample['homonym']}: {sample['judged_meaning']} "
                   f"(e.g., \"{sample['example_sentence']}\")")

  return story_part, meaning_part


def tokenize_ambistory(batch: dict) -> BatchEncoding:
    story_parts = []
    meaning_parts = []
    for i in range(len(batch["precontext"])):
        ending = batch["ending"][i] or ''
        story = f"{batch['precontext'][i]} {batch['sentence'][i]} {ending}"
        meaning = (
            f"{batch['homonym'][i]}: {batch['judged_meaning'][i]} "
            f'(e.g., "{batch["example_sentence"][i]}")'
        )
        story_parts.append(story)
        meaning_parts.append(meaning)

    return tokenizer(
        story_parts,
        meaning_parts,
        truncation="only_first",
        max_length=256,
        padding=False,
    )

def tokenize_stsb(batch: dict) -> BatchEncoding:
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation="only_first",
        max_length=256,
        padding=False
    )


def metric_computation(eval_pred: EvalPrediction) -> dict[str, float]:
  predictions, labels = eval_pred
  predictions = predictions.squeeze()

  spearman_score, _ = spearmanr(predictions, labels)

  mean_absolute_error = np.mean(np.abs(predictions-labels))
  root_mean_squared_error = np.sqrt(np.mean((predictions-labels)**2))

  return {
      "spearman": spearman_score,
      "mae": mean_absolute_error,
      "rmse": root_mean_squared_error
  }

# **Datasets**

In [6]:
ambistory_train_df = pandas.read_json("/content/train.json").T
ambistory_validation_df = pandas.read_json("/content/dev.json").T
ambistory_test_df = pandas.read_json("/content/test.json").T

In [7]:
raw_datasets = DatasetDict({
    "ambistory_train_set": Dataset.from_pandas(ambistory_train_df),
    "ambistory_validation_set": Dataset.from_pandas(ambistory_validation_df),
    "ambistory_test_set": Dataset.from_pandas(ambistory_test_df)
})

for split_name in raw_datasets:
    raw_datasets[split_name] = raw_datasets[split_name].rename_column("average", "labels")

for split_name in raw_datasets:
    raw_datasets[split_name] = raw_datasets[split_name].cast_column("labels", Value("float32"))

print(raw_datasets["ambistory_train_set"].column_names)

cols_to_remove = [c for c in raw_datasets["ambistory_train_set"].column_names if c != "labels"]

tokenized_datasets = raw_datasets.map(
    tokenize_ambistory,
    batched=True,
    remove_columns=cols_to_remove,
)

Casting the dataset:   0%|          | 0/2280 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]

['homonym', 'judged_meaning', 'precontext', 'sentence', 'ending', 'choices', 'labels', 'stdev', 'nonsensical', 'sample_id', 'example_sentence', '__index_level_0__']


Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

In [8]:
stsb = load_dataset("glue", "stsb")

stsb = stsb.rename_column("label", "labels")
stsb = stsb.map(lambda x: {"labels": float(x["labels"])})
stsb_tokenized = stsb.map(
    tokenize_stsb,
    batched=True,
    remove_columns=[c for c in stsb["train"].column_names if c != "labels"],
)

README.md: 0.00B [00:00, ?B/s]

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

Map:   0%|          | 0/5749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1379 [00:00<?, ? examples/s]

# **STS-B Pretraining**

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

stsb_args = TrainingArguments(
    output_dir="./curriculum_stsb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    seed=42,
)

stsb_trainer = Trainer(
    model=model,
    args=stsb_args,
    data_collator=data_collator,
    train_dataset=stsb_tokenized["train"],
    eval_dataset=stsb_tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

stsb_trainer.train()

stsb_trainer.save_model("./curriculum_stsb/final")

# ---- STAGE 2: AmbiStory fine-tuning on top of STS-B ----

model = AutoModelForSequenceClassification.from_pretrained(
    "./curriculum_stsb/final",
    num_labels=1,
    problem_type="regression",
)

ambistory_args = TrainingArguments(
    output_dir="./curriculum_stsb_ambistory",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=1337,
)

ambistory_trainer = Trainer(
    model=model,
    args=ambistory_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

ambistory_trainer.train()

test_metrics = ambistory_trainer.evaluate(tokenized_datasets["ambistory_test_set"])

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,0.371900,0.434820,0.921217,0.515661,0.659409
2,0.217600,0.343204,0.926191,0.451471,0.585837
3,0.118700,0.342367,0.928838,0.449586,0.585121
4,0.066900,0.310252,0.929907,0.423197,0.557003


Epoch,Training Loss,Validation Loss,Spearman,Mae,Rmse
1,0.802000,1.037452,0.616006,0.789059,1.018554
2,0.458700,1.085441,0.626792,0.805820,1.041845
3,0.371000,0.983865,0.661715,0.761111,0.991900
4,0.275700,0.965261,0.666762,0.762301,0.982477
5,0.229500,0.980077,0.665979,0.768576,0.989988


In [11]:
print(test_metrics)

{'eval_loss': 0.9686604738235474, 'eval_spearman': 0.64513699281547, 'eval_mae': 0.7680975794792175, 'eval_rmse': 0.9842054843902588, 'eval_runtime': 1.548, 'eval_samples_per_second': 600.775, 'eval_steps_per_second': 9.69, 'epoch': 5.0}


# **STS-B Contribution Check Across Different Seeds**

In [12]:
import json
import gc
import torch
import numpy as np
from pathlib import Path

# ---------- locked curriculum config ----------

STSB_CONFIG = {
    "lr": 2e-5,
    "bs": 16,
    "epochs": 4,
    "warmup": 0.1,
}

AMBISTORY_CONFIG = {
    "lr": 5e-6,
    "bs": 8,
    "epochs": 5,
    "warmup": 0.06,
}

SEEDS = [42, 1337, 2024]


def run_curriculum(seed):
    """Run STS-B → AmbiStory pipeline with given seed. Returns dev + test metrics."""

    # === STAGE 1: STS-B ===

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
    )

    stsb_args = TrainingArguments(
        output_dir=f"./curriculum_runs/stsb_seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=STSB_CONFIG["lr"],
        per_device_train_batch_size=STSB_CONFIG["bs"],
        per_device_eval_batch_size=64,
        num_train_epochs=STSB_CONFIG["epochs"],
        weight_decay=0.01,
        warmup_ratio=STSB_CONFIG["warmup"],
        bf16=True,
        load_best_model_at_end=True,
        metric_for_best_model="spearman",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=200,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    stsb_trainer = Trainer(
        model=model,
        args=stsb_args,
        data_collator=data_collator,
        train_dataset=stsb_tokenized["train"],
        eval_dataset=stsb_tokenized["validation"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    stsb_trainer.train()

    stsb_dev_spearman = stsb_trainer.state.best_metric
    print(f"  Stage 1 (STS-B): dev Spearman = {stsb_dev_spearman:.4f}")

    stage1_ckpt = f"./curriculum_runs/stsb_seed{seed}/final"
    stsb_trainer.save_model(stage1_ckpt)

    del model, stsb_trainer
    gc.collect()
    torch.cuda.empty_cache()

    # === STAGE 2: AmbiStory ===

    model = AutoModelForSequenceClassification.from_pretrained(stage1_ckpt)

    ambi_args = TrainingArguments(
        output_dir=f"./curriculum_runs/ambistory_seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=AMBISTORY_CONFIG["lr"],
        per_device_train_batch_size=AMBISTORY_CONFIG["bs"],
        per_device_eval_batch_size=64,
        num_train_epochs=AMBISTORY_CONFIG["epochs"],
        weight_decay=0.01,
        warmup_ratio=AMBISTORY_CONFIG["warmup"],
        bf16=True,
        load_best_model_at_end=True,
        metric_for_best_model="spearman",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=100,
        report_to="none",
        seed=seed,
        disable_tqdm=True,
    )

    ambi_trainer = Trainer(
        model=model,
        args=ambi_args,
        data_collator=data_collator,
        train_dataset=tokenized_datasets["ambistory_train_set"],
        eval_dataset=tokenized_datasets["ambistory_validation_set"],
        tokenizer=tokenizer,
        compute_metrics=metric_computation,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    ambi_trainer.train()

    ambi_dev_spearman = ambi_trainer.state.best_metric

    eval_logs = [
        log for log in ambi_trainer.state.log_history
        if "eval_spearman" in log and not np.isnan(log["eval_spearman"])
    ]
    best_log = max(eval_logs, key=lambda x: x["eval_spearman"])
    best_epoch = best_log["epoch"]

    test_metrics = ambi_trainer.evaluate(tokenized_datasets["ambistory_test_set"])

    test_preds = ambi_trainer.predict(tokenized_datasets["ambistory_test_set"]).predictions.squeeze()
    test_labels = ambistory_test_df["average"].to_numpy()
    test_stdev = ambistory_test_df["stdev"].to_numpy()
    threshold = np.maximum(test_stdev, 1.0)
    acc_within_sd = float(np.mean(np.abs(test_preds - test_labels) <= threshold))

    del model, ambi_trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "stsb_dev_spearman": float(stsb_dev_spearman),
        "ambi_dev_spearman": float(ambi_dev_spearman),
        "ambi_best_epoch": float(best_epoch),
        "test_spearman": float(test_metrics["eval_spearman"]),
        "test_mae": float(test_metrics["eval_mae"]),
        "test_rmse": float(test_metrics["eval_rmse"]),
        "test_acc_within_sd": acc_within_sd,
        "dev_test_gap": float(ambi_dev_spearman - test_metrics["eval_spearman"]),
    }


Path("./curriculum_runs").mkdir(exist_ok=True)
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    metrics = run_curriculum(seed)
    results[seed] = metrics

    print(f"  → STS-B dev:    {metrics['stsb_dev_spearman']:.4f}")
    print(f"  → Ambi dev:     {metrics['ambi_dev_spearman']:.4f} (best epoch {metrics['ambi_best_epoch']:.0f})")
    print(f"  → Test ρ:       {metrics['test_spearman']:.4f}")
    print(f"  → Test acc-SD:  {metrics['test_acc_within_sd']:.4f}")
    print(f"  → Dev-test gap: {metrics['dev_test_gap']:.4f}")

    with open("./curriculum_runs/multiseed_results.json", "w") as f:
        json.dump(results, f, indent=2)


print("\n\n" + "="*60)
print("MULTI-SEED CURRICULUM RESULTS (STS-B -> AmbiStory)")
print("="*60)

test_spearmans = [r["test_spearman"] for r in results.values()]
test_acc_sds = [r["test_acc_within_sd"] for r in results.values()]
dev_spearmans = [r["ambi_dev_spearman"] for r in results.values()]
gaps = [r["dev_test_gap"] for r in results.values()]
best_epochs = [r["ambi_best_epoch"] for r in results.values()]

print(f"\nTest Spearman:        {np.mean(test_spearmans):.4f} ± {np.std(test_spearmans):.4f}")
print(f"Test Acc-within-SD:   {np.mean(test_acc_sds):.4f} ± {np.std(test_acc_sds):.4f}")
print(f"AmbiStory dev ρ:      {np.mean(dev_spearmans):.4f} ± {np.std(dev_spearmans):.4f}")
print(f"Dev -> test gap:       {np.mean(gaps):.4f} ± {np.std(gaps):.4f}")
print(f"Best epoch (Stage 2): {np.mean(best_epochs):.1f} (range: {min(best_epochs):.0f}–{max(best_epochs):.0f})")

print(f"\nPer seed:")
for seed, r in results.items():
    print(f"  seed={seed}: test ρ={r['test_spearman']:.4f}, gap={r['dev_test_gap']:.4f}, best_epoch={r['ambi_best_epoch']:.0f}")

BASELINE_TEST_SPEARMAN = 0.6336
BASELINE_TEST_ACC_SD = 0.7742
BASELINE_DEV_TEST_GAP = 0.681 - 0.6336

print(f"\nVs baseline (single seed):")
print(f"  Δ Test Spearman:    {np.mean(test_spearmans) - BASELINE_TEST_SPEARMAN:+.4f}")
print(f"  Δ Test Acc-SD:      {np.mean(test_acc_sds) - BASELINE_TEST_ACC_SD:+.4f}")
print(f"  Δ Dev-test gap:     {np.mean(gaps) - BASELINE_DEV_TEST_GAP:+.4f}")


Seed 42


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.2625, 'grad_norm': 10.288454055786133, 'learning_rate': 1.9135802469135804e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.4656805396080017, 'eval_spearman': 0.9183094760219042, 'eval_mae': 0.5367740988731384, 'eval_rmse': 0.6824079155921936, 'eval_runtime': 1.5244, 'eval_samples_per_second': 983.998, 'eval_steps_per_second': 15.744, 'epoch': 1.0}
{'loss': 0.3371, 'grad_norm': 5.505843639373779, 'learning_rate': 1.6049382716049385e-05, 'epoch': 1.1111111111111112}
{'loss': 0.2185, 'grad_norm': 10.383052825927734, 'learning_rate': 1.2962962962962964e-05, 'epoch': 1.6666666666666665}
{'eval_loss': 0.4677085876464844, 'eval_spearman': 0.9259714884819011, 'eval_mae': 0.5377089381217957, 'eval_rmse': 0.6838922500610352, 'eval_runtime': 1.524, 'eval_samples_per_second': 984.264, 'eval_steps_per_second': 15.748, 'epoch': 2.0}
{'loss': 0.1808, 'grad_norm': 3.789983034133911, 'learning_rate': 9.876543209876543e-06, 'epoch': 2.2222222222222223}
{'loss': 0.1301, 'grad_norm': 6.294739

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.3583, 'grad_norm': 9.519967079162598, 'learning_rate': 1.9135802469135804e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.3329595923423767, 'eval_spearman': 0.9220800086648019, 'eval_mae': 0.438912570476532, 'eval_rmse': 0.5770265460014343, 'eval_runtime': 1.5321, 'eval_samples_per_second': 979.042, 'eval_steps_per_second': 15.665, 'epoch': 1.0}
{'loss': 0.3579, 'grad_norm': 5.073480129241943, 'learning_rate': 1.6049382716049385e-05, 'epoch': 1.1111111111111112}
{'loss': 0.2428, 'grad_norm': 9.95948600769043, 'learning_rate': 1.2962962962962964e-05, 'epoch': 1.6666666666666665}
{'eval_loss': 0.3293745517730713, 'eval_spearman': 0.9269943068894387, 'eval_mae': 0.4390903413295746, 'eval_rmse': 0.5739116072654724, 'eval_runtime': 1.5232, 'eval_samples_per_second': 984.747, 'eval_steps_per_second': 15.756, 'epoch': 2.0}
{'loss': 0.1885, 'grad_norm': 4.572856426239014, 'learning_rate': 9.876543209876543e-06, 'epoch': 2.2222222222222223}
{'loss': 0.1397, 'grad_norm': 4.405619621

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 2.0304, 'grad_norm': 19.066776275634766, 'learning_rate': 1.9135802469135804e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.37625667452812195, 'eval_spearman': 0.9218875696595947, 'eval_mae': 0.47293123602867126, 'eval_rmse': 0.6133976578712463, 'eval_runtime': 1.5459, 'eval_samples_per_second': 970.339, 'eval_steps_per_second': 15.525, 'epoch': 1.0}
{'loss': 0.3426, 'grad_norm': 9.62915325164795, 'learning_rate': 1.6049382716049385e-05, 'epoch': 1.1111111111111112}
{'loss': 0.2302, 'grad_norm': 5.918331146240234, 'learning_rate': 1.2962962962962964e-05, 'epoch': 1.6666666666666665}
{'eval_loss': 0.3276771605014801, 'eval_spearman': 0.9299140702797493, 'eval_mae': 0.4507625997066498, 'eval_rmse': 0.5724309086799622, 'eval_runtime': 1.5377, 'eval_samples_per_second': 975.49, 'eval_steps_per_second': 15.608, 'epoch': 2.0}
{'loss': 0.193, 'grad_norm': 5.235779285430908, 'learning_rate': 9.876543209876543e-06, 'epoch': 2.2222222222222223}
{'loss': 0.1274, 'grad_norm': 6.6623535

In [14]:
# === V2 ABLATION: shorter STS-B ===

STSB_CONFIG_V2 = {
    "lr": 2e-5,
    "bs": 16,
    "epochs": 2,
    "warmup": 0.1,
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

stsb_args_v2 = TrainingArguments(
    output_dir="./curriculum_v2/stsb_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=STSB_CONFIG_V2["lr"],
    per_device_train_batch_size=STSB_CONFIG_V2["bs"],
    per_device_eval_batch_size=64,
    num_train_epochs=STSB_CONFIG_V2["epochs"],
    weight_decay=0.01,
    warmup_ratio=STSB_CONFIG_V2["warmup"],
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
    seed=42,
    disable_tqdm=True,
)

stsb_trainer_v2 = Trainer(
    model=model,
    args=stsb_args_v2,
    data_collator=data_collator,
    train_dataset=stsb_tokenized["train"],
    eval_dataset=stsb_tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
)

stsb_trainer_v2.train()
print(f"Stage 1 (2 epochs) STS-B dev: {stsb_trainer_v2.state.best_metric:.4f}")
stsb_trainer_v2.save_model("./curriculum_v2/stsb_seed42/final")

del model, stsb_trainer_v2
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained("./curriculum_v2/stsb_seed42/final")

ambi_args_v2 = TrainingArguments(
    output_dir="./curriculum_v2/ambistory_seed42",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to="none",
    seed=42,
    disable_tqdm=True,
)

ambi_trainer_v2 = Trainer(
    model=model,
    args=ambi_args_v2,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["ambistory_train_set"],
    eval_dataset=tokenized_datasets["ambistory_validation_set"],
    tokenizer=tokenizer,
    compute_metrics=metric_computation,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

ambi_trainer_v2.train()

test_metrics = ambi_trainer_v2.evaluate(tokenized_datasets["ambistory_test_set"])
test_preds = ambi_trainer_v2.predict(tokenized_datasets["ambistory_test_set"]).predictions.squeeze()
test_labels = ambistory_test_df["average"].to_numpy()
test_stdev = ambistory_test_df["stdev"].to_numpy()
threshold = np.maximum(test_stdev, 1.0)
acc_within_sd = float(np.mean(np.abs(test_preds - test_labels) <= threshold))

print(f"\n=== V2 Ablation (STS-B 2 epochs, seed=42) ===")
print(f"  Test ρ:       {test_metrics['eval_spearman']:.4f}")
print(f"  Test acc-SD:  {acc_within_sd:.4f}")
print(f"  Dev-test gap: {ambi_trainer_v2.state.best_metric - test_metrics['eval_spearman']:.4f}")
print(f"\nReference numbers:")
print(f"  Baseline (single seed): test ρ=0.6336")
print(f"  STS-B v1 mean (3 seeds): test ρ=0.617 ± 0.006")
print(f"  STS-B v2 seed=42:       test ρ={test_metrics['eval_spearman']:.4f}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/deberta-v3-large-zeroshot-v2.0 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([1, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 1.8236, 'grad_norm': 10.964653015136719, 'learning_rate': 1.6049382716049385e-05, 'epoch': 0.5555555555555556}
{'eval_loss': 0.42056527733802795, 'eval_spearman': 0.9086176837969095, 'eval_mae': 0.4921852946281433, 'eval_rmse': 0.6485099792480469, 'eval_runtime': 1.5309, 'eval_samples_per_second': 979.811, 'eval_steps_per_second': 15.677, 'epoch': 1.0}
{'loss': 0.3513, 'grad_norm': 7.462048053741455, 'learning_rate': 9.876543209876543e-06, 'epoch': 1.1111111111111112}
{'loss': 0.217, 'grad_norm': 6.169938087463379, 'learning_rate': 3.7037037037037037e-06, 'epoch': 1.6666666666666665}
{'eval_loss': 0.34636059403419495, 'eval_spearman': 0.9233200343963102, 'eval_mae': 0.44844916462898254, 'eval_rmse': 0.5885241031646729, 'eval_runtime': 1.5623, 'eval_samples_per_second': 960.139, 'eval_steps_per_second': 15.362, 'epoch': 2.0}
{'train_runtime': 193.0644, 'train_samples_per_second': 59.555, 'train_steps_per_second': 3.729, 'train_loss': 0.6972854746712579, 'epoch': 2.0}
Stage 1 (2